In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Modul 3 – Perbaikan Citra dan Deteksi Tepi
## 1. Import Library

Pada tahap awal, dilakukan import terhadap tiga library inti:

| Library | Peran dalam Praktikum |
|---|---|
| **`cv2` (OpenCV)** | Membaca gambar (`imread`), konversi ruang warna (`cvtColor`), operasi morfologi, dan Gaussian blur |
| **`matplotlib.pyplot`** | Menampilkan citra dan histogram secara visual dalam format subplot |
| **`numpy`** | Komputasi numerik – operasi matriks, pembuatan kernel, padding, dan clipping nilai piksel |

Ketiga library ini menjadi fondasi pipeline pengolahan citra digital yang dibangun dalam modul ini.

In [ ]:
img_bgr  = cv2.imread('backup.jpg')
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print('Grayscale shape :', img_gray.shape)

plt.imshow(img_gray, cmap='gray')
plt.title('Grayscale Image – backup.jpg (Citra CCTV)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Konversi Citra ke Grayscale

### Tujuan
Citra warna (BGR) memiliki **3 kanal** (Blue, Green, Red) dengan total informasi 3× lebih besar.
Konversi ke **grayscale** mereduksinya menjadi **1 kanal intensitas** sehingga:

- Komputasi filter dan konvolusi menjadi **jauh lebih ringan**.
- Deteksi tepi bekerja berdasarkan **perbedaan intensitas** (bukan warna), sehingga 1 kanal sudah cukup.
- Operator gradien (Sobel, Prewitt, Roberts) dirancang untuk citra **2-D**.

### Formula Konversi
OpenCV menggunakan formula standar ITU-R BT.601:

```
Gray = 0.114·B + 0.587·G + 0.299·R
```

Bobot berbeda karena mata manusia paling sensitif terhadap cahaya **hijau**, lalu merah, dan paling sedikit biru.

### Hasil pada Citra CCTV
Citra `backup.jpg` merupakan rekaman CCTV kualitas rendah. Setelah konversi ke grayscale, noise pada background (goresan vertikal) sudah terlihat jelas. Ini menjadi motivasi utama untuk melakukan **perbaikan kualitas** (smoothing) sebelum proses deteksi tepi.

In [ ]:
def filter(img, size, mode):
    """Filter spasial: mean, median, atau modus."""
    height, width = img.shape
    pad = size // 2
    padded = np.pad(img, pad, mode='edge')
    canvas = np.zeros_like(img, dtype=np.uint8)
    match mode:
        case 'mean':
            area = size * size
            for i in range(height):
                for j in range(width):
                    region = padded[i:i+size, j:j+size]
                    canvas[i, j] = np.sum(region) / area
        case 'median':
            for i in range(height):
                for j in range(width):
                    region = padded[i:i+size, j:j+size]
                    canvas[i, j] = np.median(region)
        case 'modus':
            for i in range(height):
                for j in range(width):
                    region = padded[i:i+size, j:j+size]
                    values = region.ravel()
                    count = {}
                    for val in values:
                        count[val] = count.get(val, 0) + 1
                    canvas[i, j] = max(count, key=count.get)
    return canvas


def convolution(img, kernel):
    """Konvolusi 2-D manual dengan zero-padding."""
    size = kernel.shape[0]
    pad_size = size // 2
    padded = np.pad(img, pad_size, mode='constant')
    canvas = np.zeros_like(img, dtype=np.float32)
    height, width = img.shape
    for i in range(height):
        for j in range(width):
            region = padded[i:i+size, j:j+size]
            canvas[i, j] = np.sum(region * kernel)
    return canvas


def edge(img, kernelx, kernely):
    """Hitung magnitudo gradien: |Gx| + |Gy|, lalu normalisasi 0-255."""
    gx = convolution(img, kernelx)
    gy = convolution(img, kernely)
    canvas = np.abs(gx) + np.abs(gy)
    canvas = canvas * 255.0 / (np.max(canvas) + 1e-9)
    return np.clip(canvas, 0, 255).astype(np.uint8)

In [ ]:
# ── Kernel Smoothing & Sharpening ────────────────────────────────────────────
kernelSmoothing = np.array([
    [1/10, 1/10, 1/10],
    [1/10, 1/5,  1/10],
    [1/10, 1/10, 1/10]
])

kernelSharpening = np.array([
    [1/9, 1/9, 1/9],
    [1/9, 8/9, 1/9],
    [1/9, 1/9, 1/9]
])

# ── Kernel Deteksi Tepi ───────────────────────────────────────────────────────
sobelX = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32)
sobelY = np.array([[ 1,2,1],[ 0,0,0],[-1,-2,-1]], dtype=np.float32)

prewittX = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32)
prewittY = np.array([[ 1,1,1],[ 0,0,0],[-1,-1,-1]], dtype=np.float32)

robertsX = np.array([[1, 0],[0,-1]], dtype=np.float32)
robertsY = np.array([[0, 1],[-1,0]], dtype=np.float32)

print("Semua kernel berhasil didefinisikan.")

In [ ]:
smoothing   = convolution(img_gray, kernelSmoothing)
sharpening  = convolution(img_gray, kernelSharpening)
smoothsharp = convolution(smoothing, kernelSharpening)

# Normalisasi ke uint8 agar dapat ditampilkan
def to_disp(arr):
    arr = arr - arr.min()
    arr = arr / (arr.max() + 1e-9) * 255
    return np.clip(arr, 0, 255).astype(np.uint8)

plt.figure(figsize=(15, 5))
for idx, (title, img) in enumerate(
        [('Smoothing', smoothing), ('Sharpening', sharpening), ('Smoothing + Sharpening', smoothsharp)], 1):
    plt.subplot(1, 3, idx)
    plt.imshow(to_disp(img), cmap='gray')
    plt.title(title)
    plt.axis('off')
plt.suptitle('Perbandingan Smoothing, Sharpening, dan Kombinasi (backup.jpg)', fontsize=13)
plt.tight_layout()
plt.show()

# ── Perbandingan Mean vs Median vs Modus ─────────────────────────────────────
print("Menghitung filter statistik (mungkin lambat karena loop Python)...")
mean_img   = filter(img_gray, 3, 'mean')
median_img = filter(img_gray, 3, 'median')
modus_img  = filter(img_gray, 3, 'modus')

plt.figure(figsize=(15, 5))
for idx, (title, img) in enumerate(
        [('Mean Filter', mean_img), ('Median Filter', median_img), ('Modus Filter', modus_img)], 1):
    plt.subplot(1, 3, idx)
    plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
plt.suptitle('Perbandingan Mean vs Median vs Modus (window 3×3, backup.jpg)', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Analisis: Smoothing, Sharpening, dan Kombinasinya

---

### A. Perbedaan Tiga Metode

#### 1. Smoothing (Penghalusan)
- **Tujuan**: Mengurangi noise dan detail halus yang mengganggu dengan meratakan nilai piksel bertetangga.
- **Cara Kerja**: Kernel yang digunakan memberi bobot lebih besar ke piksel pusat (1/5) dan bobot lebih kecil ke sekeliling (1/10). Operasi konvolusi mengambil rata-rata berbobot dari sekitar setiap piksel.
- **Efek Visual**: Gambar terlihat *blur* atau buram. Garis-garis noise halus pada citra CCTV berkurang.
- **Kegunaan**: Pra-pemrosesan (pre-processing) sebelum deteksi tepi agar noise tidak ikut terdeteksi sebagai tepi.

#### 2. Sharpening (Penajaman)
- **Tujuan**: Menonjolkan tepi dan detail yang ada dalam citra.
- **Cara Kerja**: Kernel menggunakan nilai tengah sangat besar (8/9) dan nilai sekitarnya positif kecil (1/9). Ini berbeda dari Laplacian yang memiliki nilai negatif di sekitarnya; kernel ini bersifat *unsharp masking ringan* yang memperkuat piksel pusat relatif terhadap sekitarnya.
- **Efek Visual**: Kontras pada batas-batas objek meningkat. Tepi tampak lebih tegas.
- **Kegunaan**: Meningkatkan keterbacaan detail pada citra, misalnya untuk identifikasi wajah pada rekaman CCTV.

#### 3. Smoothing + Sharpening (Kombinasi Berurutan)
- **Tujuan**: Menggabungkan keunggulan keduanya – menghilangkan noise terlebih dahulu, baru kemudian mempertajam tepi yang tersisa.
- **Cara Kerja**: Citra dihaluskan dulu (noise berkurang), lalu hasil smoothing diproses sharpening. Tepi yang dipertajam adalah tepi "bersih" tanpa noise.
- **Efek Visual**: Tepi objek utama (misalnya kontur tubuh) lebih tegas, namun noise halus background tidak ikut dipertajam.
- **Kegunaan**: Pipeline terbaik untuk citra kualitas rendah seperti CCTV. Hasil lebih informatif dibandingkan sharpening langsung pada citra ber-noise.

| Aspek | Smoothing | Sharpening | Smooth + Sharp |
|---|---|---|---|
| Noise | Dikurangi ✓ | Diperkuat ✗ | Dikurangi dulu ✓ |
| Detail Tepi | Melemah | Diperkuat ✓ | Diperkuat bersih ✓ |
| Urutan proses | Pre-processing | Enhancement | Kombinasi optimal |

---

### B. Perbedaan Mean, Median, dan Modus

Ketiga metode ini adalah **filter statistik** yang bekerja pada area (window) sekitar setiap piksel:

#### Mean Filter (Rata-Rata)
- **Cara**: Piksel output = rata-rata aritmetika semua piksel dalam window.
- **Kelebihan**: Cepat secara komputasi, mengurangi noise Gaussian secara efektif.
- **Kekurangan**: **Memudarkan tepi** (blurring) dan **tidak tahan terhadap noise Salt-and-Pepper** – piksel outlier yang ekstrem langsung menarik rata-rata.
- **Hasil**: Blur merata ke seluruh citra.

#### Median Filter
- **Cara**: Piksel output = nilai tengah (median) dari semua piksel dalam window setelah diurutkan.
- **Kelebihan**: **Sangat baik untuk noise Salt-and-Pepper** karena nilai ekstrem tidak mempengaruhi median. Tepi relatif lebih terjaga dibanding mean.
- **Kekurangan**: Sedikit lebih lambat dari mean (perlu sorting).
- **Hasil**: Noise titik hilang, tepi lebih tajam dibanding hasil mean.

#### Modus Filter (Mode)
- **Cara**: Piksel output = nilai yang paling sering muncul (frekuensi tertinggi) dalam window.
- **Kelebihan**: Mempertahankan piksel yang "mayoritas" di sekitar area tertentu; cocok untuk citra dengan area homogen luas.
- **Kekurangan**: Tidak stabil jika distribusi nilai piksel merata (banyak nilai unik) – hasilnya bisa acak atau tidak konsisten. Paling lambat karena perlu menghitung frekuensi.
- **Hasil**: Pada citra ber-noise acak, hasilnya sering tidak lebih baik dari median, dan pada area gradien halus bisa menghasilkan artefak kotak (posterisasi).

| Filter | Tahan Salt-Pepper | Mempertajam Tepi | Kecepatan |
|---|---|---|---|
| Mean | ✗ | ✗ (blur) | Cepat |
| Median | ✓ | Lebih baik dari mean | Sedang |
| Modus | Tergantung distribusi | Tidak konsisten | Lambat |

> **Kesimpulan**: Untuk citra CCTV yang mengandung noise acak, **Median Filter** adalah pilihan terbaik karena mampu membersihkan noise sekaligus mempertahankan tepi subjek.

In [ ]:
def to_uint8(img):
    return np.clip(img, 0, 255).astype(np.uint8)

def normalize(img):
    mn, mx = img.min(), img.max()
    if mx == mn:
        return np.zeros_like(img, dtype=np.uint8)
    return to_uint8((img - mn) / (mx - mn) * 255)

In [ ]:
hasil_tepiSobel   = edge(img_gray, sobelX, sobelY)
hasil_tepiPrewitt = edge(img_gray, prewittX, prewittY)
hasil_tepiRoberts = edge(img_gray, robertsX, robertsY)

sobpre    = normalize(hasil_tepiSobel   + hasil_tepiPrewitt)
prerob    = normalize(hasil_tepiPrewitt + hasil_tepiRoberts)
sobrob    = normalize(hasil_tepiSobel   + hasil_tepiRoberts)
presobrob = normalize(hasil_tepiPrewitt + hasil_tepiSobel + hasil_tepiRoberts)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
data = [
    ('Prewitt',                hasil_tepiPrewitt),
    ('Sobel',                  hasil_tepiSobel),
    ('Roberts',                hasil_tepiRoberts),
    ('Prewitt + Sobel',        sobpre),
    ('Prewitt + Roberts',      prerob),
    ('Sobel + Roberts',        sobrob),
    ('Prewitt + Sobel + Roberts', presobrob),
]
for ax, (title, img) in zip(axes.flatten(), data):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=11)
    ax.axis('off')
axes.flatten()[-1].axis('off')   # cell kosong

plt.suptitle('Deteksi Tepi: Operator Tunggal vs Kombinasi (backup.jpg)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Analisis: Operator Deteksi Tepi dan Kombinasinya

---

### A. Penjelasan Setiap Operator

#### 1. Operator Prewitt
- **Kernel**: Koefisien seragam (−1, 0, +1) di setiap baris/kolom kernel.
- **Cara Kerja**: Menghitung gradien horizontal (Gx) dan vertikal (Gy) secara terpisah, lalu magnitudonya dijumlah: |Gx| + |Gy|.
- **Sifat**: Memberikan **bobot yang sama** pada semua piksel tetangga dalam arah gradien.
- **Kelebihan**: Implementasi sederhana, komputasi ringan.
- **Kekurangan**: Lebih sensitif terhadap noise dibanding Sobel karena tidak ada pembobotan terhadap piksel terdekat.
- **Hasil pada CCTV**: Mendeteksi tepi kontur tubuh dan latar cukup merata, namun noise tipis tetap ikut terdeteksi.

#### 2. Operator Sobel
- **Kernel**: Baris/kolom tengah diberi bobot **2×** lebih besar (−2, 0, +2 untuk baris tengah).
- **Cara Kerja**: Sama seperti Prewitt – gradien Gx dan Gy – namun piksel yang lebih dekat ke pusat mendapat pengaruh lebih besar.
- **Sifat**: Mengandung **pembobotan Gaussian implisit** sehingga lebih tahan noise.
- **Kelebihan**: Tepi lebih tegas dan tebal, lebih robust terhadap noise dibanding Prewitt.
- **Kekurangan**: Tepi yang dihasilkan sedikit lebih lebar (piksel tepi lebih banyak).
- **Hasil pada CCTV**: Kontur wajah dan tubuh lebih tajam dibanding Prewitt; garis vertikal latar background juga terdeteksi jelas.

#### 3. Operator Roberts
- **Kernel**: Hanya berukuran **2×2**, menghitung gradien diagonal (bukan horizontal-vertikal).
  - Gx: `[[1,0],[0,−1]]`, Gy: `[[0,1],[−1,0]]`
- **Cara Kerja**: Mendeteksi perubahan diagonal pada piksel bertetangga langsung.
- **Sifat**: Sangat sensitif terhadap perubahan kecil (presisi tinggi), tetapi **sangat rentan noise**.
- **Kelebihan**: Menghasilkan tepi yang **tipis dan tepat** pada citra bersih.
- **Kekurangan**: Kernel kecil membuat operator tidak memiliki efek penghalusan sama sekali – noise langsung terdeteksi sebagai tepi.
- **Hasil pada CCTV**: Tepi sangat tipis dan sebagian besar tersembunyi di noise; kurang cocok untuk citra CCTV berkualitas rendah.

---

### B. Analisis Kombinasi Operator

Kombinasi dilakukan dengan **menjumlahkan nilai magnitudo** dari dua atau lebih operator, lalu dinormalisasi ke rentang 0–255. Setiap operator menangkap "sudut pandang" yang berbeda tentang tepi yang ada.

#### Prewitt + Sobel
- Keduanya mendeteksi gradien horizontal dan vertikal, namun dengan pembobotan berbeda.
- **Efek**: Tepi semakin tebal dan kuat. Area yang hanya terdeteksi lemah oleh salah satu operator menjadi lebih jelas.
- Cocok jika ingin memastikan semua tepi "mayor" terdeteksi.

#### Prewitt + Roberts
- Gabungan gradien seragam (Prewitt) dengan gradien diagonal sensitif (Roberts).
- **Efek**: Tepi diagonal kecil yang tidak terdeteksi Prewitt akan muncul. Namun noise juga ikut meningkat karena Roberts.

#### Sobel + Roberts
- Kombinasi yang saling melengkapi: Sobel memberikan tepi robust dan tebal, Roberts menambah detail diagonal halus.
- **Efek**: Tepi horizontal/vertikal kuat dari Sobel, ditambah aksen diagonal dari Roberts. Noise cukup terkendalikan oleh dominasi Sobel.

#### Prewitt + Sobel + Roberts (Triple Combination)
- Semua tepi dari tiga perspektif digabungkan.
- **Efek**: Citra tepi paling "penuh" – hampir semua transisi intensitas terdeteksi, namun noise juga paling tinggi.
- Berguna jika diikuti oleh thresholding yang tepat untuk menyaring noise.

| Kombinasi | Tepi Terdeteksi | Noise | Cocok Untuk |
|---|---|---|---|
| Prewitt saja | Sedang | Rendah-Sedang | Analisis umum |
| Sobel saja | Kuat, tebal | Rendah | Citra ber-noise |
| Roberts saja | Tipis, presisi | Tinggi | Citra bersih saja |
| Prewitt + Sobel | Kuat | Rendah | Tepi mayor |
| Sobel + Roberts | Kuat + Diagonal | Sedang | Keseimbangan |
| Triple | Terlengkap | Tinggi | Perlu threshold ketat |

> **Kesimpulan**: Untuk citra CCTV `backup.jpg`, **Operator Sobel tunggal atau Sobel + Prewitt** memberikan hasil terbaik karena menghasilkan tepi yang kuat dan cukup tahan terhadap noise background. Roberts sebaiknya dihindari atau hanya dikombinasikan setelah smoothing.

In [ ]:
thresholds = [10, 20, 30, 40, 50, 60]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, t in zip(axes.flatten(), thresholds):
    binary = np.where(hasil_tepiSobel >= t, 255, 0).astype(np.uint8)
    white_pct = (binary == 255).sum() / binary.size * 100
    ax.imshow(binary, cmap='gray')
    ax.set_title(f'Threshold {t}  |  Tepi: {white_pct:.1f}%', fontsize=11)
    ax.axis('off')

plt.suptitle('Perbandingan Nilai Threshold pada Hasil Sobel Edge Detection', fontsize=13)
plt.tight_layout()
plt.show()

# Tampilkan statistik piksel tepi untuk setiap threshold
print(f"{'Threshold':>12} | {'Jumlah Piksel Tepi':>20} | {'Persentase (%)':>15}")
print('-' * 55)
h, w = hasil_tepiSobel.shape
total = h * w
for t in thresholds:
    cnt = (hasil_tepiSobel >= t).sum()
    print(f"{t:>12} | {cnt:>20,} | {cnt/total*100:>14.2f}%")

## 5. Analisis: Pengaruh Nilai Threshold pada Deteksi Tepi

---

### A. Cara Kerja Thresholding

Setelah deteksi tepi menghasilkan **citra gradien** (nilai 0–255), thresholding mengubahnya menjadi **citra biner**:

```
piksel_output = 255  jika nilai_gradien >= T
piksel_output = 0    jika nilai_gradien < T
```

Di mana **T** adalah nilai ambang batas (threshold). Semakin tinggi T, semakin sedikit piksel yang lolos sebagai "tepi".

---

### B. Perbedaan Mencolok Threshold Rendah vs Tinggi

#### Threshold Rendah (T = 10)
- **Karakteristik**: Hampir semua piksel dengan perubahan intensitas sekecil apapun dianggap sebagai tepi.
- **Visual**: Citra tepi sangat **ramai dan putih** – penuh dengan garis-garis, termasuk noise dari background CCTV (goresan vertikal), tekstur dinding, dan detail halus yang tidak signifikan.
- **Masalah**: **Noise dianggap tepi (false positive tinggi)**. Sulit membedakan tepi objek asli dengan artefak gambar.
- **Kegunaan**: Hanya cocok untuk citra sangat bersih tanpa noise.

#### Threshold Tinggi (T = 50–60)
- **Karakteristik**: Hanya piksel dengan perubahan intensitas sangat drastis yang dianggap tepi.
- **Visual**: Citra tepi sangat **gelap dan sparse** – hanya sedikit garis yang terlihat, kemungkinan hanya kontur terkuat saja.
- **Masalah**: **Detail penting hilang (false negative tinggi)**. Tepi halus seperti batas wajah dan pakaian bisa tidak terdeteksi sama sekali.
- **Kegunaan**: Cocok jika hanya ingin mendeteksi struktur besar yang sangat kontras.

---

### C. Rekomendasi Nilai Threshold Optimal

**Threshold terbaik untuk kasus citra CCTV `backup.jpg`: T = 20 atau T = 30**

**Alasan untuk T = 20:**
- Kontur tubuh orang masih terdeteksi dengan jelas dan cukup lengkap.
- Noise halus pada background berkurang signifikan dibandingkan T = 10.
- Tepi mayor (batas kepala, bahu, lengan) masih terlihat sebagai garis yang terhubung.
- Keseimbangan antara *detail yang terjaga* dan *noise yang tersaring*.

**Alasan untuk T = 30 (sebagai alternatif):**
- Noise semakin bersih; hanya tepi dengan kontras tinggi yang tersisa.
- Cocok jika tujuannya adalah **identifikasi siluet kasar** bukan detail halus.
- Berisiko kehilangan beberapa tepi penting (seperti batas rambut atau detail wajah).

| Threshold | Evaluasi | Rekomendasi |
|---|---|---|
| 10 | Terlalu banyak noise, tidak bisa dibaca | ✗ |
| 20 | Keseimbangan detail & kebersihan **optimal** | ✓ Terbaik |
| 30 | Bersih, tepi mayor terjaga, sedikit detail hilang | ✓ Alternatif |
| 40 | Mulai kehilangan informasi penting | ~ |
| 50 | Hanya siluet sangat kasar | ✗ |
| 60 | Hampir tidak ada tepi terdeteksi | ✗ |

> **Kesimpulan**: Nilai threshold **T = 20** memberikan hasil paling informatif untuk citra CCTV ini – cukup menyaring noise namun masih mempertahankan tepi yang relevan untuk analisis bentuk subjek.

In [ ]:
# ── 1. Persiapan Citra & ROI ──────────────────────────────────────
img       = cv2.imread("backup.jpg")
backuprgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
backup    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
tinggi, lebar = backup.shape

x1, y1, x2, y2 = 190, 35, 380, 450
roi = backup[y1:y2, x1:x2]

# ── 2. Blur + Double Thresholding ─────────────────────────────────
roi_blur         = cv2.GaussianBlur(roi, (5, 5), 0)
_, thresh_dark   = cv2.threshold(roi_blur, 0,   255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
_, thresh_bright = cv2.threshold(roi_blur, 160, 255, cv2.THRESH_BINARY)
roi_thresh       = cv2.bitwise_or(thresh_dark, thresh_bright)

# ── 3. Morfologi & Fill Kontur Terbesar ───────────────────────────
kernel9    = np.ones((9, 9), np.uint8)
roi_closed = cv2.morphologyEx(roi_thresh, cv2.MORPH_CLOSE, kernel9, iterations=5)
roi_opened = cv2.morphologyEx(roi_closed, cv2.MORPH_OPEN,  kernel9, iterations=2)

roi_mask  = np.zeros_like(roi_opened)
contours, _ = cv2.findContours(roi_opened, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
if contours:
    terbesar = max(contours, key=cv2.contourArea)
    cv2.drawContours(roi_mask, [terbesar], -1, 255, thickness=cv2.FILLED)

# ── 4. Kembalikan ke Kanvas Asli & Bersihkan ──────────────────────
mask = np.zeros((tinggi, lebar), dtype=np.uint8)
mask[y1:y2, x1:x2] = roi_mask
mask = cv2.dilate(mask, kernel9, iterations=1)
mask[y1:y1+120, 340:] = 0
mask[:y1, :] = 0

# ── 5. Outline & Pewarnaan ────────────────────────────────────────
outline = cv2.morphologyEx(mask, cv2.MORPH_GRADIENT, np.ones((5, 5), np.uint8))

hasil = backuprgb.copy().astype(np.int16)
badan = (mask == 255)
hasil[badan, 0] = np.clip(hasil[badan, 0] + 80, 0, 255)
hasil[badan, 1] = np.clip(hasil[badan, 1] + 80, 0, 255)
hasil[badan, 2] = np.clip(hasil[badan, 2] - 60, 0, 255)
hasil[outline > 0] = [255, 255, 80]
hasil = hasil.astype(np.uint8)

# ── 6. Visualisasi ────────────────────────────────────────────────
plt.figure(figsize=(18, 6))
plt.subplot(1, 3, 1); plt.imshow(mask, cmap='gray');   plt.title('Mask Final (Siluet)'); plt.axis('off')
plt.subplot(1, 3, 2); plt.imshow(outline, cmap='gray'); plt.title('Outline Kontur');      plt.axis('off')
plt.subplot(1, 3, 3); plt.imshow(hasil);                plt.title('Highlight Kuning pada Citra Asli'); plt.axis('off')
plt.suptitle('CCTV: Highlight Siluet Orang', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Analisis: Highlight Siluet Orang pada Citra CCTV

---

### Tujuan
Menonjolkan area tubuh manusia pada rekaman CCTV agar bentuk subjek lebih mudah dikenali secara visual, tanpa memerlukan model machine learning.

### Pipeline yang Digunakan

```
Citra Grayscale (ROI)
       ↓
Gaussian Blur (5×5)          → Mengurangi noise agar threshold lebih stabil
       ↓
Double Thresholding           → thresh_dark (Otsu) | thresh_bright (160)
   → bitwise_OR              → Menangkap area gelap (pakaian) DAN area terang (kulit/wajah)
       ↓
Morfologi CLOSE (9×9, ×5)    → Menutup celah kecil dalam siluet
Morfologi OPEN  (9×9, ×2)    → Menghapus noise kecil yang terisolasi
       ↓
findContours → Ambil Blob Terbesar → drawContours (FILLED)
                                  → Mengisi seluruh area dalam kontur = mask solid
       ↓
Dilasi + Pembersihan Manual  → Memperluas mask sedikit, hapus area salah
       ↓
MORPH_GRADIENT               → Outline = erosi dikurangi dilasi = tepi tipis kontur
       ↓
Pewarnaan: Area badan R+80,G+80,B−60 → Highlight kekuningan
           Outline: [255,255,80]     → Garis kuning terang
```

### Mengapa Masking (Bukan Deteksi Tepi Langsung)?
Deteksi tepi menghasilkan garis tipis yang tidak "mengisi" area objek. Untuk **highlight area penuh** (bukan hanya kontur), diperlukan **mask biner** yang menutup seluruh piksel dalam area subjek. Itulah fungsi kombinasi thresholding + morfologi + flood fill via `findContours`.

### Kelebihan Pendekatan Ini
- Tidak memerlukan model deep learning atau dataset.
- Cepat diimplementasikan dengan OpenCV.
- Dapat disesuaikan (ROI, threshold, warna highlight) sesuai kebutuhan.

### Keterbatasan
- ROI (Region of Interest) harus ditentukan secara manual – tidak otomatis mendeteksi lokasi orang.
- Double thresholding sensitif terhadap perubahan pencahayaan.
- Hasil belum sempurna karena background CCTV yang kompleks (motif vertikal mirip baju).

In [ ]:
# ── Muat Citra david.jpg ─────────────────────────────────────────
david_bgr  = cv2.imread('david.jpg')
david_rgb  = cv2.cvtColor(david_bgr, cv2.COLOR_BGR2RGB)
david_gray = cv2.cvtColor(david_bgr, cv2.COLOR_BGR2GRAY)

# ── Deteksi Tepi (Sobel) ──────────────────────────────────────────
david_edge = edge(david_gray, sobelX, sobelY)

# ── Segmentasi Background Toska via Kanal RGB ────────────────────
R = david_rgb[:,:,0].astype(np.int32)
G = david_rgb[:,:,1].astype(np.int32)
B = david_rgb[:,:,2].astype(np.int32)

# Logika: toska dominan G dan B, sementara R rendah
bg_mask = ((G > R + 20) & (G > 80) & (B > R - 30)).astype(np.uint8) * 255
fg_mask = 255 - bg_mask   # foreground = kebalikan background

# ── Hapus Background (ganti dengan putih) ────────────────────────
result = david_rgb.copy()
result[bg_mask == 255] = [255, 255, 255]

# ── Visualisasi Lengkap ───────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

axes[0,0].imshow(david_rgb);              axes[0,0].set_title('Citra Asli (david.jpg)');           axes[0,0].axis('off')
axes[0,1].imshow(david_gray, cmap='gray');axes[0,1].set_title('Grayscale');                        axes[0,1].axis('off')
axes[0,2].imshow(david_edge, cmap='gray');axes[0,2].set_title('Deteksi Tepi (Sobel)');             axes[0,2].axis('off')
axes[1,0].imshow(bg_mask, cmap='gray');   axes[1,0].set_title('Mask Background (toska)');          axes[1,0].axis('off')
axes[1,1].imshow(fg_mask, cmap='gray');   axes[1,1].set_title('Mask Foreground (subjek)');         axes[1,1].axis('off')
axes[1,2].imshow(result);                 axes[1,2].set_title('Hasil: Background Dihapus (putih)');axes[1,2].axis('off')

plt.suptitle('Pipeline Deteksi Tepi + Background Removal – david.jpg', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Analisis: Pas Foto – Deteksi Tepi dan Penghilangan Background

---

### A. Mengapa Pas Foto Berbeda dari Citra CCTV?

| Aspek | CCTV (backup.jpg) | Pas Foto (david.jpg) |
|---|---|---|
| Pencahayaan | Rendah, tidak merata | Baik, seragam |
| Noise | Tinggi (grain + artefak kompresi) | Rendah |
| Background | Kompleks (motif vertikal) | Sederhana (warna solid toska) |
| Tepi subjek | Sulit dibedakan dari BG | Kontras tinggi |

Kondisi yang lebih baik pada `david.jpg` membuat deteksi tepi **jauh lebih efektif** dan penghilangan background **dapat dilakukan dengan pendekatan sederhana berbasis warna**.

---

### B. Proses Deteksi Tepi pada david.jpg

**Langkah:**
1. Konversi BGR → Grayscale (satu kanal intensitas).
2. Terapkan konvolusi Sobel (Gx dan Gy) untuk menghitung gradien.
3. Magnitudo: `|Gx| + |Gy|`, normalisasi ke 0–255.

**Hasil:**
- Tepi wajah (dahi, pipi, dagu, alis) terdeteksi sebagai garis putih tegas.
- Batas rambut terlihat jelas karena kontras tinggi antara rambut gelap dan background toska terang.
- Kontur baju (kaos hitam) terdeteksi di bagian bawah.

Kualitas pencahayaan yang baik terbukti menghasilkan **tepi yang tajam dan informatif** tanpa perlu pre-processing smoothing intensif.

---

### C. Proses Penghilangan Background (Background Removal)

#### Strategi: Pemisahan Kanal RGB
Background `david.jpg` berwarna **toska** (teal/cyan) – warna yang merupakan campuran Hijau dan Biru dengan sedikit Merah.

**Karakteristik warna toska dalam ruang RGB:**
- Nilai R rendah
- Nilai G tinggi
- Nilai B relatif tinggi

**Kondisi logis yang digunakan:**
```python
bg_mask = (G > R + 20) AND (G > 80) AND (B > R - 30)
```

| Kondisi | Makna |
|---|---|
| `G > R + 20` | Hijau dominan dibanding Merah (ciri khas toska/cyan) |
| `G > 80` | Pastikan area cukup terang, bukan bayangan gelap |
| `B > R - 30` | Biru tidak jauh lebih rendah dari Merah (saring warna hangat) |

**Mengapa bukan deteksi tepi untuk BG removal?**
Deteksi tepi hanya menghasilkan **garis**, bukan area tertutup. Untuk menghapus seluruh region background, diperlukan **segmentasi berbasis warna** yang menghasilkan mask area penuh.

#### Langkah-langkah:
```
1. Baca citra asli RGB
2. Pisahkan kanal R, G, B
3. Terapkan logika kondisional → bg_mask (piksel background = 255)
4. fg_mask = NOT(bg_mask) → piksel foreground = 255
5. Salin citra asli → result
6. Semua piksel pada result yang bg_mask == 255 diganti [255,255,255]
7. Area foreground dipertahankan nilai aslinya
```

#### Hasil:
- Subjek (wajah + rambut + baju) tetap utuh dengan warna asli.
- Background toska digantikan dengan warna putih bersih.
- Tidak ada proses morfologi atau machine learning – cukup operasi bitwise berbasis warna.

---

### D. Keterbatasan Pendekatan Ini
- Jika piksel kulit atau pakaian kebetulan memenuhi kondisi toska (misal: ada warna teal di baju), piksel tersebut akan ikut dihapus.
- Tidak robust untuk background dengan warna yang mirip kulit atau bervariasi.
- Untuk hasil lebih baik, dapat dikombinasikan dengan median filtering pada mask dan operasi morfologi untuk menghaluskan tepian mask.

---

## 8. Kesimpulan Akhir: Perbaikan Citra dan Deteksi Tepi dalam Pengolahan Citra Digital

---

Berdasarkan seluruh proses yang telah dilakukan pada Modul 3, dapat ditarik beberapa kesimpulan penting:

### 1. Perbaikan Kualitas Citra adalah Fondasi Wajib
Tanpa pre-processing, deteksi tepi pada citra berkualitas rendah (seperti CCTV) menghasilkan output yang dipenuhi noise. **Smoothing (terutama Median Filter)** terbukti efektif mengurangi noise sambil mempertahankan tepi objek. Pipeline **Smooth → Sharp** menghasilkan citra yang paling siap untuk analisis lanjutan.

### 2. Pemilihan Operator Deteksi Tepi Harus Disesuaikan dengan Kualitas Citra
- **Sobel**: Pilihan terbaik untuk citra ber-noise karena memiliki pembobotan yang menghaluskan secara implisit.
- **Prewitt**: Baik untuk analisis umum pada citra sedang.
- **Roberts**: Hanya cocok untuk citra bersih dan beresolusi tinggi.
- **Kombinasi operator** dapat meningkatkan kelengkapan tepi yang terdeteksi, namun harus diikuti threshold yang tepat.

### 3. Nilai Threshold Menentukan Kualitas Hasil Deteksi
Threshold adalah parameter kritis: **terlalu rendah → noise terdeteksi, terlalu tinggi → tepi penting hilang**. Nilai optimal (T = 20–30 untuk citra CCTV ini) harus ditentukan berdasarkan karakteristik spesifik citra dan tujuan analisis.

### 4. Segmentasi Berbasis Warna Lebih Efektif dari Deteksi Tepi untuk Background Removal
Pada `david.jpg`, pemisahan kanal RGB memungkinkan penghapusan background toska secara akurat tanpa perlu model kompleks. Ini menunjukkan bahwa **pengetahuan tentang karakteristik citra** (warna background diketahui) memungkinkan solusi sederhana namun efektif.

### 5. Masking + Morfologi untuk Highlight Area Objek
Untuk menonjolkan area subjek secara visual (highlight siluet CCTV), pendekatan berbasis mask lebih tepat dibanding deteksi tepi langsung. Thresholding ganda + morfologi + fill kontur menghasilkan mask area penuh yang dapat diwarnai sesuai kebutuhan.

### Ringkasan Pipeline Terbaik untuk Setiap Kasus:

| Kasus | Pipeline yang Direkomendasikan |
|---|---|
| Citra CCTV ber-noise | Grayscale → Median Filter → Sobel Edge → Threshold T=20 |
| Pas foto bersih | Grayscale → Sobel Edge → RGB Segmentation → BG Removal |
| Highlight siluet | Gaussian Blur → Double Threshold → Morfologi → Contour Fill → Mask Coloring |
| Identifikasi tepi detail | Smoothing → Sobel + Prewitt kombinasi → Threshold adaptif |

> Pengolahan citra digital bukan satu solusi untuk semua kasus – pemilihan metode yang tepat bergantung pada **kualitas input**, **tujuan analisis**, dan **karakteristik objek** yang ingin diekstrak.